# Truss equilibrium

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
import numpy as np
from scipy.linalg import qr, solve_triangular
from compas.datastructures import Graph
from compas_notebook import Viewer
from compas_notebook.config import Config

## Truss graph

In [ ]:
span, rise, P = 4.0, 2.0, 10.0
truss = Graph()
for key, xyz in {"left": [0, 0, 0], "top": [span / 2, 0, rise], "right": [span, 0, 0]}.items():
    truss.add_node(key, x=xyz[0], y=xyz[1], z=xyz[2], px=0.0, pz=0.0)
truss.node_attribute("top", "pz", -P)
members = [("left", "top"), ("top", "right"), ("left", "right")]
for u, v in members:
    truss.add_edge(u, v)
supports = [("left", 0), ("left", 1), ("right", 1)]

## Equilibrium matrix

In [ ]:
row = truss.node_index()
E = np.zeros((2 * truss.number_of_nodes(), len(members) + len(supports)))
for j, (u, v) in enumerate(members):
    d = np.subtract(truss.node_coordinates(v), truss.node_coordinates(u))[[0, 2]]
    d /= np.linalg.norm(d)
    E[2 * row[u]:2 * row[u] + 2, j] = d
    E[2 * row[v]:2 * row[v] + 2, j] = -d
for j, (key, axis) in enumerate(supports, start=len(members)):
    E[2 * row[key] + axis, j] = 1.0
loads = np.array([truss.node_attributes(key, ["px", "pz"]) for key in truss.nodes()]).ravel()
E

## QR solve

In [ ]:
Q, R = qr(E)
unknowns = solve_triangular(R, Q.T @ -loads)
forces, reactions = unknowns[:len(members)], unknowns[len(members):]
for edge, force in zip(members, forces):
    truss.edge_attribute(edge, "force", float(force))
forces, reactions

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [5, -8, 4]
config.view.camera.target = [span / 2, 0, rise / 2]
viewer = Viewer(config=config)
viewer.scene.add(truss, color="#5552bb", nodesize=0.10)
viewer.show()